# Farm Detection — Image Classifier Training Notebook

**Project:** plant & animal disease, pest and weed detection for farmers (Kenya / Kilifi focus)
**Author:** Timothy Masha · Diploma in Computer Science, Pwani University TVET College

This notebook takes the data in `data/farm_detection_dataset.csv` and the photos in `data/images/`
and goes all the way to a trained model your app can use:

| Step | What happens |
|---|---|
| 0 | Setup: Colab GPU, paths, libraries, random seeds |
| 1 | Settings you can change (image size, model, which categories to train) |
| 2 | *(Optional)* Download extra open datasets: beans (iBean) and cassava (iCassava) |
| 3 | Load the CSV and build one table with one row per image |
| 4 | Data audit: broken images, duplicates, leakage between splits, class balance, image statistics |
| 5 | Pick classes, cap very large classes, compute class weights |
| 6 | Preprocessing pipeline: resize, augmentation, batching |
| 7 | Model: transfer learning with EfficientNetV2 |
| 8 | Training stage 1 (frozen backbone) and stage 2 (fine-tuning) |
| 9 | Testing and statistics: accuracy, precision/recall/F1, confusion matrix, Cohen's kappa, MCC, ROC-AUC, bootstrap confidence intervals, calibration, McNemar test, lab-vs-field comparison |
| 10 | Confidence threshold for "not sure — call an expert" |
| 11 | Diagnosis demo: photo → disease → treatments from the CSV |
| 12 | Export: Keras model, TFLite model for the phone app, labels, reports — saved to Google Drive |

### How to run on Google Colab (free GPU)
1. Upload **`FarmDetection_Project_part1.zip`** and **`FarmDetection_Project_part2.zip`** to the top of your
   Google Drive (`My Drive`). Don't unzip them; the notebook does that.
2. Open this notebook in Colab: *File → Upload notebook*.
3. *Runtime → Change runtime type → **T4 GPU***.
4. *Runtime → Run all*. Accept the Google Drive permission pop-up.

Expect roughly 30–60 minutes on a T4 GPU with the default settings. Results are copied back to
`My Drive/FarmDetection_outputs/` at the end so nothing is lost when Colab disconnects.

## 0. Setup

In [ ]:
import os, sys, json, math, random, hashlib, time, shutil, zipfile, re, csv
from collections import Counter

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    import glob
    # The project comes as one or more zip parts (FarmDetection_Project_part1.zip, _part2.zip ...).
    # Put all of them in "My Drive"; they unzip into the same folder.
    DRIVE_ZIPS = sorted(glob.glob("/content/drive/MyDrive/FarmDetection_Project*.zip"))
    PROJECT = "/content/FarmDetection_Project"
    if not os.path.exists(os.path.join(PROJECT, ".unzipped")):
        if not DRIVE_ZIPS:
            raise FileNotFoundError("Upload FarmDetection_Project_part1.zip and _part2.zip to My Drive first")
        for zp in DRIVE_ZIPS:
            print("Unzipping", os.path.basename(zp), "(a few minutes the first time)...")
            with zipfile.ZipFile(zp) as z:
                z.extractall("/content")
        open(os.path.join(PROJECT, ".unzipped"), "w").close()
    OUTPUT_ON_DRIVE = "/content/drive/MyDrive/FarmDetection_outputs"
else:
    # Running on your own computer: start Jupyter from the project folder or the notebooks folder
    here = os.getcwd()
    PROJECT = os.path.dirname(here) if os.path.basename(here) == "notebooks" else here
    OUTPUT_ON_DRIVE = None

DATA = os.path.join(PROJECT, "data")
CSV_PATH = os.path.join(DATA, "farm_detection_dataset.csv")
MODELS_DIR = os.path.join(PROJECT, "models")
REPORTS_DIR = os.path.join(PROJECT, "reports")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)
sys.path.insert(0, os.path.join(PROJECT, "scripts"))
print("Project folder:", PROJECT)
print("CSV found:", os.path.exists(CSV_PATH))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn import metrics as skm
from sklearn.utils.class_weight import compute_class_weight
from scipy import stats

pd.set_option("display.max_colwidth", 80)
plt.rcParams["figure.dpi"] = 110

try:
    import tensorflow as tf
    from tensorflow import keras
    print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
    gpus = tf.config.list_physical_devices("GPU")
    print("GPU:", gpus if gpus else "NONE - go to Runtime > Change runtime type > T4 GPU")
except ImportError:
    tf = None
    print("TensorFlow is not installed. Install it (pip install tensorflow) or run on Google Colab.")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
if tf is not None:
    tf.random.set_seed(SEED)

## 1. Settings you can change

* **`CATEGORIES_TO_TRAIN`** — which kinds of records become classes. Crop diseases, crop pests and
  healthy crops (the `crop_requirement` rows) are trained together by default.
  Animals and soils look completely different, so train them as **separate models** once you
  have photos: set this to `["animal_disease", "animal_pest"]` or `["soil_type"]` and run again.
* **`MIN_TRAIN_IMAGES`** — classes with fewer training photos than this are skipped (and listed).
* **`MAX_TRAIN_PER_CLASS`** — very large classes (e.g. 5,000 citrus-greening photos) are randomly
  reduced in *training* so they don't dominate. Validation and test sets are never reduced.

In [ ]:
CATEGORIES_TO_TRAIN = ["crop_disease", "crop_pest", "crop_requirement", "weed"]
MODEL_NAME = "farm_crop_model"          # used in output file names

IMG_SIZE = 224
BATCH_SIZE = 32
BACKBONE = "EfficientNetV2B0"           # or "MobileNetV3Large" (smaller, faster on phones)
MIN_TRAIN_IMAGES = 30
MAX_TRAIN_PER_CLASS = 1500
EPOCHS_HEAD = 8                         # stage 1: only the new top layers learn
EPOCHS_FINE = 12                        # stage 2: fine-tune the top of the backbone
FINE_TUNE_FRACTION = 0.4                # share of backbone layers unfrozen in stage 2
LR_HEAD, LR_FINE = 1e-3, 1e-4
USE_TFDS_EXTRAS = True                  # download beans + cassava photos (needs internet, ~1.5 GB)
TARGET_ACCURACY_FOR_THRESHOLD = 0.95    # used to choose the "not sure" confidence threshold
N_BOOTSTRAP = 1000

## 2. *(Optional)* Download extra open datasets

These come from TensorFlow Datasets and are added straight into the right record folders:

| Dataset | Classes → CSV records |
|---|---|
| **iBean** (Makerere AI Lab, MIT licence) | angular leaf spot → CD008, bean rust → CD009, healthy → CR002 |
| **iCassava 2019** (Makerere AI Lab) | CBB → CD028, CBSD → CD027, green mite → CP027, CMD → CD026, healthy → CR005 |

A marker file stops it downloading twice. Photos are named `<ID>_ibean_0001.jpg` / `<ID>_icassava_0001.jpg`.

In [ ]:
def add_tfds_dataset(tfds_name, mapping, source):
    import tensorflow_datasets as tfds
    import arrange_open_datasets as aod
    marker = os.path.join(DATA, f".{source}_done")
    if os.path.exists(marker):
        print(f"{tfds_name}: already added")
        return
    ds, info = tfds.load(tfds_name, split="train+validation+test", with_info=True, as_supervised=True)
    names = info.features["label"].names
    next_num, folders, added = {}, {}, Counter()
    for img, lab in tfds.as_numpy(ds):
        rid = mapping.get(names[int(lab)])
        if rid is None:
            continue
        if rid not in folders:
            folders[rid] = aod.folder_for(rid)
            os.makedirs(folders[rid], exist_ok=True)
            next_num[rid] = aod.next_number(folders[rid], rid, source)
        im = Image.fromarray(img).convert("RGB")
        im.thumbnail((512, 512))
        im.save(os.path.join(folders[rid], f"{rid}_{source}_{next_num[rid]:04d}.jpg"), "JPEG", quality=90)
        next_num[rid] += 1
        added[rid] += 1
    open(marker, "w").close()
    print(f"{tfds_name}: added {dict(added)}")

if USE_TFDS_EXTRAS and tf is not None:
    from dataset_mapping import TFDS_BEANS, TFDS_CASSAVA
    for name, mapping, source in [("beans", TFDS_BEANS, "ibean"), ("cassava", TFDS_CASSAVA, "icassava")]:
        try:
            add_tfds_dataset(name, mapping, source)
        except Exception as e:
            print(f"Could not add {name}: {e}  (training continues without it)")

# Refresh the CSV image columns (also picks up any of your own new photos)
import update_images
_ = update_images.run()

## 3. Load the CSV and build the image table

The CSV has one row per record (disease, pest, weed, soil, crop) with its image folder and photo
counts. Here we list the photos in every folder and give each one its split
(train / validation / test) with the same fixed rule used by `scripts/update_images.py`, giving
**one row per image**.

In [ ]:
from update_images import list_record_images, source_of

records = pd.read_csv(CSV_PATH, dtype=str).fillna("")
records["image_count"] = records["image_count"].replace("", "0").astype(int)
print(f"{len(records)} records in the CSV")
display(records.groupby("category")["image_count"].agg(records="count", images="sum"))

def display_name(row):
    """Healthy-crop classes come from crop_requirement rows."""
    if row["category"] == "crop_requirement":
        return f"Healthy {row['name']}"
    return row["name"]

rows = []
for _, r in records.iterrows():
    folder = os.path.join(DATA, "images", r["image_folder"])
    for fname, split in list_record_images(folder, r["id"]):
        rows.append({"path": os.path.join(folder, fname), "record_id": r["id"], "class_name": display_name(r),
                     "category": r["category"], "source": source_of(fname), "split": split})
images = pd.DataFrame(rows)
print(f"{len(images):,} images found")
display(images.groupby("split").size().rename("images"))
display(images.groupby("source").size().rename("images"))

## 4. Data audit

Before training we check the data, because a model is only as good as its photos.

### 4.1 Missing and broken images

In [ ]:
def check_image(path):
    try:
        with Image.open(path) as im:
            im.verify()
        return True
    except Exception:
        return False

t0 = time.time()
images["exists"] = images["path"].map(os.path.exists)
images["readable"] = False
images.loc[images["exists"], "readable"] = images.loc[images["exists"], "path"].map(check_image)
bad = images[~images["readable"]]
print(f"Checked {len(images):,} images in {time.time()-t0:.0f}s -> {len(bad)} missing/broken")
if len(bad):
    display(bad.head(20))
images = images[images["readable"]].drop(columns=["exists", "readable"]).reset_index(drop=True)

### 4.2 Duplicate photos and data leakage

If the *same* photo is in both training and test, the test score is falsely high (leakage).
We hash every file; for exact duplicates we keep one copy, preferring the train split.

In [ ]:
def md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

images["md5"] = images["path"].map(md5)
dup_groups = images[images.duplicated("md5", keep=False)]
cross_split = dup_groups.groupby("md5")["split"].nunique()
cross_split = cross_split[cross_split > 1]
cross_label = dup_groups.groupby("md5")["record_id"].nunique()
cross_label = cross_label[cross_label > 1]
print(f"Duplicate files: {images.duplicated('md5').sum()}")
print(f"  ...of which appear in more than one split (leakage): {len(cross_split)}")
print(f"  ...of which have conflicting labels: {len(cross_label)}")

split_rank = {"train": 0, "val": 1, "test": 2}
images = (images.assign(_r=images["split"].map(split_rank))
                .sort_values(["md5", "_r"]).drop_duplicates("md5", keep="first")
                .drop(columns="_r").sort_index().reset_index(drop=True))
images = images[~images["md5"].isin(cross_label.index)].reset_index(drop=True)
print(f"Images after removing duplicates: {len(images):,}")

### 4.3 Class balance

In [ ]:
counts = images.pivot_table(index="class_name", columns="split", values="path", aggfunc="count", fill_value=0)
counts = counts.reindex(columns=["train", "val", "test"], fill_value=0)
counts["total"] = counts.sum(axis=1)
counts = counts.sort_values("total")
ax = counts[["train", "val", "test"]].plot.barh(stacked=True, figsize=(9, max(4, 0.32 * len(counts))))
ax.set_xlabel("number of images"); ax.set_title("Images per class and split")
plt.tight_layout(); plt.show()

tot = counts["total"]
print(f"Classes with images: {len(tot)}")
print(f"Smallest class: {tot.min()} ({tot.idxmin()}), largest: {tot.max()} ({tot.idxmax()})")
print(f"Imbalance ratio (largest / smallest): {tot.max() / tot.min():.1f}")
print(f"Coefficient of variation of class sizes: {tot.std() / tot.mean():.2f}")

In [ ]:
src = images.pivot_table(index="class_name", columns="source", values="path", aggfunc="count", fill_value=0)
ax = (src.div(src.sum(axis=1), axis=0)).plot.barh(stacked=True, figsize=(9, max(4, 0.32 * len(src))))
ax.set_xlabel("share of images"); ax.set_title("Where each class's photos come from")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left"); plt.tight_layout(); plt.show()

### 4.4 Image statistics (sample)

Lab photos (PlantVillage: single leaf, plain background) look very different from field photos
(PlantDoc, iCassava, your own). The numbers below show that difference — it is why we later
test the model **separately on field photos**.

In [ ]:
sample = pd.concat([d.sample(min(len(d), 150), random_state=SEED) for _, d in images.groupby("source")])
stats_rows = []
for _, r in sample.iterrows():
    with Image.open(r["path"]) as im:
        w, h = im.size
        arr = np.asarray(im.convert("RGB").resize((64, 64)), dtype=np.float32) / 255.0
    stats_rows.append({"source": r["source"], "width": w, "height": h,
                       "mean_R": arr[..., 0].mean(), "mean_G": arr[..., 1].mean(), "mean_B": arr[..., 2].mean(),
                       "brightness_std": arr.mean(axis=2).std()})
img_stats = pd.DataFrame(stats_rows)
display(img_stats.groupby("source").agg(["mean", "std"]).round(3))

# Is brightness variation different between sources? (one-way ANOVA + Kruskal-Wallis)
groups = [g["brightness_std"].values for _, g in img_stats.groupby("source")]
if len(groups) > 1:
    f, p = stats.f_oneway(*groups)
    h, p2 = stats.kruskal(*groups)
    print(f"ANOVA F={f:.1f}, p={p:.2g} | Kruskal-Wallis H={h:.1f}, p={p2:.2g}")
    print("A small p (< 0.05) means the sources really differ in image contrast.")

### 4.5 Look at the photos

In [ ]:
def show_grid(df, n_cols=6, title=None, label_fn=None, size=2.2):
    n = len(df)
    n_rows = max(1, math.ceil(n / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(size * n_cols, size * 1.15 * n_rows))
    axes = np.array(axes).reshape(-1)
    for ax in axes:
        ax.axis("off")
    for ax, (_, r) in zip(axes, df.iterrows()):
        with Image.open(r["path"]) as im:
            ax.imshow(im.convert("RGB"))
        ax.set_title(label_fn(r) if label_fn else f"{r['class_name'][:22]}\n({r['source']})", fontsize=7)
    if title:
        fig.suptitle(title)
    plt.tight_layout(); plt.show()

one_per_class = images.groupby("class_name").sample(1, random_state=SEED)
show_grid(one_per_class, title="One example per class")

## 5. Choose classes, balance the training set, class weights

In [ ]:
use = images[images["category"].isin(CATEGORIES_TO_TRAIN)].copy()
train_counts = use[use["split"] == "train"].groupby("record_id").size()
keep_ids = sorted(train_counts[train_counts >= MIN_TRAIN_IMAGES].index)
skipped = sorted(set(use["record_id"]) - set(keep_ids))
if skipped:
    print("Skipped (fewer than", MIN_TRAIN_IMAGES, "training photos):", skipped)
use = use[use["record_id"].isin(keep_ids)].reset_index(drop=True)

# Class index order = sorted record IDs (stable between runs)
class_ids = keep_ids
id_to_idx = {rid: i for i, rid in enumerate(class_ids)}
id_to_name = use.drop_duplicates("record_id").set_index("record_id")["class_name"].to_dict()
class_names = [id_to_name[r] for r in class_ids]
use["label"] = use["record_id"].map(id_to_idx)
NUM_CLASSES = len(class_ids)
print(f"Training {NUM_CLASSES} classes:")
for i, (rid, nm) in enumerate(zip(class_ids, class_names)):
    print(f"  {i:2d}  {rid}  {nm}")

# Cap very large classes in TRAIN only (stratified by source so field photos are kept)
def cap_class(d):
    if len(d) <= MAX_TRAIN_PER_CLASS:
        return d
    field = d[d["source"] != "plantvillage"]
    lab = d[d["source"] == "plantvillage"]
    n_lab = max(0, MAX_TRAIN_PER_CLASS - len(field))
    return pd.concat([field, lab.sample(min(len(lab), n_lab), random_state=SEED)])

train_df = pd.concat([cap_class(d) for _, d in use[use["split"] == "train"].groupby("record_id")])
val_df = use[use["split"] == "val"]
test_df = use[use["split"] == "test"]
train_df = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f"\ntrain {len(train_df):,} | val {len(val_df):,} | test {len(test_df):,}")

cw = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=train_df["label"].values)
class_weight = {i: float(w) for i, w in enumerate(cw)}
display(pd.DataFrame({"record_id": class_ids, "class": class_names,
                      "train_images": train_df["label"].value_counts().reindex(range(NUM_CLASSES)).values,
                      "class_weight": np.round(cw, 2)}))

labels_info = [{"index": i, "record_id": rid, "class_name": nm} for i, (rid, nm) in enumerate(zip(class_ids, class_names))]
with open(os.path.join(MODELS_DIR, f"{MODEL_NAME}_labels.json"), "w") as f:
    json.dump(labels_info, f, indent=2)

## 6. Preprocessing pipeline

* **Decode & resize** every photo to `IMG_SIZE × IMG_SIZE`.
* **Pixel scaling** is done inside EfficientNetV2/MobileNetV3 (they expect values 0–255), so the
  phone app can feed raw pixels too.
* **Augmentation (training only):** random flips, rotation, zoom, shift, brightness and contrast —
  this imitates different phone angles and light in the field and reduces overfitting.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return tf.cast(img, tf.float32), label

augment = keras.Sequential([
    keras.layers.RandomFlip("horizontal_and_vertical"),
    keras.layers.RandomRotation(0.15),
    keras.layers.RandomZoom((-0.15, 0.15)),
    keras.layers.RandomTranslation(0.1, 0.1),
    keras.layers.RandomBrightness(0.2, value_range=(0, 255)),
    keras.layers.RandomContrast(0.2),
], name="augmentation")

def make_ds(df, training):
    ds = tf.data.Dataset.from_tensor_slices((df["path"].values, df["label"].values.astype("int32")))
    if training:
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (tf.clip_by_value(augment(x, training=True), 0, 255), y), num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_ds(train_df, training=True)
val_ds = make_ds(val_df, training=False)
test_ds = make_ds(test_df, training=False)

# Show what augmentation does to one photo
x, _ = load_image(train_df["path"].iloc[0], 0)
fig, axes = plt.subplots(1, 6, figsize=(13, 2.6))
for i, ax in enumerate(axes):
    img = x if i == 0 else tf.clip_by_value(augment(x[None], training=True)[0], 0, 255)
    ax.imshow(img.numpy().astype("uint8")); ax.axis("off"); ax.set_title("original" if i == 0 else f"augmented {i}", fontsize=8)
plt.show()

## 7. Model: transfer learning

We start from a network already trained on 1.2 million ImageNet photos, remove its top, and add a
new classifier for our classes. Stage 1 trains only the new top; stage 2 also fine-tunes the upper
part of the backbone with a small learning rate.

In [ ]:
def build_model():
    if BACKBONE == "EfficientNetV2B0":
        base = keras.applications.EfficientNetV2B0(include_top=False, weights="imagenet",
                                                    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_preprocessing=True)
    elif BACKBONE == "MobileNetV3Large":
        base = keras.applications.MobileNetV3Large(include_top=False, weights="imagenet",
                                                    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_preprocessing=True)
    else:
        raise ValueError(BACKBONE)
    base.trainable = False
    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
    x = base(inputs, training=False)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(NUM_CLASSES, activation="softmax", name="probs")(x)
    return keras.Model(inputs, outputs, name=MODEL_NAME), base

model, base = build_model()
model.summary(show_trainable=True)

def compile_model(lr):
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss=keras.losses.SparseCategoricalCrossentropy(),
                  metrics=["accuracy", keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3")])

## 8. Training
### 8.1 Stage 1 — train the new classifier (backbone frozen)

In [ ]:
compile_model(LR_HEAD)
ckpt1 = os.path.join(MODELS_DIR, f"{MODEL_NAME}_stage1.keras")
cb1 = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
       keras.callbacks.ModelCheckpoint(ckpt1, monitor="val_loss", save_best_only=True),
       keras.callbacks.CSVLogger(os.path.join(REPORTS_DIR, "history_stage1.csv"))]
t0 = time.time()
hist1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, class_weight=class_weight, callbacks=cb1)
print(f"Stage 1 took {(time.time()-t0)/60:.1f} min")

# Keep stage-1 predictions on the test set: used later as the baseline in the McNemar test
probs_stage1 = model.predict(test_ds, verbose=0)

### 8.2 Stage 2 — fine-tune the top of the backbone

In [ ]:
base.trainable = True
n_freeze = int(len(base.layers) * (1 - FINE_TUNE_FRACTION))
for i, layer in enumerate(base.layers):
    # keep lower layers and ALL BatchNormalization layers frozen (standard practice for small data)
    layer.trainable = i >= n_freeze and not isinstance(layer, keras.layers.BatchNormalization)
print(f"Unfrozen backbone layers: {sum(l.trainable for l in base.layers)} of {len(base.layers)}")

compile_model(LR_FINE)
ckpt2 = os.path.join(MODELS_DIR, f"{MODEL_NAME}_best.keras")
cb2 = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
       keras.callbacks.ModelCheckpoint(ckpt2, monitor="val_loss", save_best_only=True),
       keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6),
       keras.callbacks.CSVLogger(os.path.join(REPORTS_DIR, "history_stage2.csv"))]
t0 = time.time()
hist2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FINE, class_weight=class_weight, callbacks=cb2)
print(f"Stage 2 took {(time.time()-t0)/60:.1f} min")

### 8.3 Learning curves
Training and validation lines should go together. If validation loss rises while training loss
keeps falling, the model is **overfitting** (memorising) — early stopping handles that.

In [ ]:
h = {k: hist1.history[k] + hist2.history.get(k, []) for k in hist1.history}
ep = np.arange(1, len(h["loss"]) + 1)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, key in zip(axes, ["loss", "accuracy", "top3"]):
    ax.plot(ep, h[key], label="train"); ax.plot(ep, h[f"val_{key}"], label="validation")
    ax.axvline(len(hist1.history["loss"]) + 0.5, color="grey", ls="--", lw=1)
    ax.set_title(key); ax.set_xlabel("epoch"); ax.legend()
axes[0].text(len(hist1.history["loss"]) + 0.7, max(h["loss"]) * 0.95, "fine-tuning starts", fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(REPORTS_DIR, "learning_curves.png")); plt.show()

## 9. Testing and statistics

The **test set** was never used for training or for choosing settings, so it gives an honest
estimate of how the model performs on new photos.

### 9.1 Overall metrics

In [ ]:
probs = model.predict(test_ds, verbose=0)
y_true = test_df["label"].values
y_pred = probs.argmax(1)
y_pred1 = probs_stage1.argmax(1)
top3_hit = np.any(np.argsort(-probs, 1)[:, :3] == y_true[:, None], axis=1)

def summary_metrics(y_true, y_pred, probs=None):
    out = {
        "accuracy": skm.accuracy_score(y_true, y_pred),
        "balanced_accuracy": skm.balanced_accuracy_score(y_true, y_pred),
        "macro_precision": skm.precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": skm.recall_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_f1": skm.f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": skm.f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "cohen_kappa": skm.cohen_kappa_score(y_true, y_pred),
        "matthews_corrcoef": skm.matthews_corrcoef(y_true, y_pred),
    }
    if probs is not None:
        out["log_loss"] = skm.log_loss(y_true, probs, labels=np.arange(probs.shape[1]))
        present = np.unique(y_true)
        if len(present) == probs.shape[1]:
            out["macro_roc_auc_ovr"] = skm.roc_auc_score(y_true, probs, multi_class="ovr", average="macro")
    return out

overall = summary_metrics(y_true, y_pred, probs)
overall["top3_accuracy"] = top3_hit.mean()
display(pd.Series(overall, name="test set").round(4).to_frame())

### 9.2 Bootstrap 95% confidence intervals
A single accuracy number hides uncertainty. We resample the test set 1,000 times with replacement
and recompute the metrics; the middle 95% of the results is the confidence interval.

In [ ]:
def bootstrap_ci(y_true, y_pred, fn, n=N_BOOTSTRAP, seed=SEED):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(y_true))
    vals = []
    for _ in range(n):
        s = rng.choice(idx, size=len(idx), replace=True)
        vals.append(fn(y_true[s], y_pred[s]))
    return np.percentile(vals, [2.5, 97.5])

ci_rows = []
for name, fn in [("accuracy", skm.accuracy_score),
                 ("macro_f1", lambda a, b: skm.f1_score(a, b, average="macro", zero_division=0)),
                 ("balanced_accuracy", skm.balanced_accuracy_score),
                 ("cohen_kappa", skm.cohen_kappa_score)]:
    lo, hi = bootstrap_ci(y_true, y_pred, fn)
    ci_rows.append({"metric": name, "value": fn(y_true, y_pred), "ci_95_low": lo, "ci_95_high": hi})
ci_table = pd.DataFrame(ci_rows).round(4)
display(ci_table)

### 9.3 Per-class results
* **Precision** — when the model says "late blight", how often is it right?
* **Recall (sensitivity)** — of all real late-blight photos, how many did it catch?
* **Wilson 95% CI** on recall — wide intervals mean too few test photos for that class.

In [ ]:
def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    d = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / d
    m = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return (c - m, c + m)

prec, rec, f1, sup = skm.precision_recall_fscore_support(y_true, y_pred, labels=np.arange(NUM_CLASSES), zero_division=0)
per_class = pd.DataFrame({"record_id": class_ids, "class": class_names, "precision": prec,
                          "recall": rec, "f1": f1, "test_images": sup})
cis = [wilson_ci(int(round(r * n)), int(n)) for r, n in zip(rec, sup)]
per_class["recall_ci_low"] = [c[0] for c in cis]
per_class["recall_ci_high"] = [c[1] for c in cis]
# specificity = true negatives / all negatives, per class
cm = skm.confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
tn = cm.sum() - cm.sum(0) - cm.sum(1) + np.diag(cm)
per_class["specificity"] = tn / (tn + cm.sum(0) - np.diag(cm))
per_class = per_class.sort_values("f1")
display(per_class.round(3))
per_class.to_csv(os.path.join(REPORTS_DIR, "per_class_metrics.csv"), index=False)

### 9.4 Confusion matrix
Rows = true class, columns = predicted class (row-normalised, so each row sums to 1).
Bright squares off the diagonal show which diseases get mixed up.

In [ ]:
cm_norm = cm / cm.sum(1, keepdims=True).clip(min=1)
fig, ax = plt.subplots(figsize=(0.45 * NUM_CLASSES + 4, 0.45 * NUM_CLASSES + 3))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
short = [f"{rid} {nm[:18]}" for rid, nm in zip(class_ids, class_names)]
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(short, rotation=90, fontsize=7)
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(short, fontsize=7)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if cm_norm[i, j] >= 0.01:
            ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center", fontsize=6,
                    color="white" if cm_norm[i, j] > 0.5 else "black")
ax.set_xlabel("predicted"); ax.set_ylabel("true"); fig.colorbar(im, fraction=0.03)
plt.tight_layout(); plt.savefig(os.path.join(REPORTS_DIR, "confusion_matrix.png")); plt.show()
pd.DataFrame(cm, index=class_ids, columns=class_ids).to_csv(os.path.join(REPORTS_DIR, "confusion_matrix.csv"))

pairs = [(class_names[i], class_names[j], cm[i, j]) for i in range(NUM_CLASSES) for j in range(NUM_CLASSES) if i != j and cm[i, j] > 0]
display(pd.DataFrame(sorted(pairs, key=lambda t: -t[2])[:10], columns=["true", "predicted as", "count"]))

### 9.5 Lab photos vs field photos (domain shift)
Farmers will take field photos, so the **field rows are the numbers that matter for the app**.

In [ ]:
test_eval = test_df.assign(correct=(y_pred == y_true), pred=y_pred)
by_source = []
for s, d in test_eval.groupby("source"):
    k, n = int(d["correct"].sum()), len(d)
    lo, hi = wilson_ci(k, n)
    by_source.append({"source": s, "test_images": n, "accuracy": k / n, "ci_95_low": lo, "ci_95_high": hi,
                      "macro_f1": skm.f1_score(d["label"], d["pred"], average="macro", zero_division=0)})
by_source = pd.DataFrame(by_source)
display(by_source.round(3))

lab = test_eval[test_eval["source"] == "plantvillage"]["correct"]
field = test_eval[test_eval["source"] != "plantvillage"]["correct"]
if len(lab) and len(field) and (~test_eval["correct"]).any():
    table = np.array([[lab.sum(), (~lab).sum()], [field.sum(), (~field).sum()]])
    chi2, p, _, _ = stats.chi2_contingency(table)
    print(f"Lab accuracy {lab.mean():.3f} vs field accuracy {field.mean():.3f}: chi-square={chi2:.1f}, p={p:.2g}")
    print("p < 0.05 means the drop on field photos is real, not chance -> collect more field photos.")

### 9.6 Did fine-tuning really help? (McNemar test)
Both models are tested on the *same* photos, so we compare them with McNemar's test, which only
looks at photos where exactly one model was right.

In [ ]:
a = (y_pred1 == y_true); b = (y_pred == y_true)
n01 = int(np.sum(a & ~b))   # stage 1 right, stage 2 wrong
n10 = int(np.sum(~a & b))   # stage 1 wrong, stage 2 right
p_mc = stats.binomtest(n10, n01 + n10, 0.5).pvalue if (n01 + n10) else 1.0
print(f"Stage 1 accuracy {a.mean():.4f} -> stage 2 accuracy {b.mean():.4f}")
print(f"Only stage 2 right: {n10} | only stage 1 right: {n01} | exact McNemar p = {p_mc:.3g}")
print("Fine-tuning made a statistically significant difference." if p_mc < 0.05 else "No significant difference.")

### 9.7 Calibration — can we trust the confidence numbers?
If the model says "90% sure", it should be right about 90% of the time. **ECE** (expected
calibration error) measures the gap; 0 is perfect. Points on the diagonal = well calibrated.

In [ ]:
def calibration(probs, y_true, n_bins=15):
    conf = probs.max(1); correct = probs.argmax(1) == y_true
    bins = np.linspace(0, 1, n_bins + 1)
    ece, xs, ys, ns = 0.0, [], [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
            xs.append(conf[m].mean()); ys.append(correct[m].mean()); ns.append(m.sum())
    return ece, xs, ys, ns

ece, xs, ys, ns = calibration(probs, y_true)
fig, ax = plt.subplots(figsize=(4.2, 4))
ax.plot([0, 1], [0, 1], "--", color="grey"); ax.plot(xs, ys, "o-")
ax.set_xlabel("model confidence"); ax.set_ylabel("actual accuracy"); ax.set_title(f"Reliability diagram (ECE = {ece:.3f})")
plt.tight_layout(); plt.savefig(os.path.join(REPORTS_DIR, "calibration.png")); plt.show()

## 10. "Not sure — call an expert" threshold
The app should not guess when unsure. We pick the **lowest confidence** at which predictions
above it are at least `TARGET_ACCURACY_FOR_THRESHOLD` accurate (chosen on the *validation* set,
then checked on the test set). Below it, the app tells the farmer to contact an extension officer or vet.

In [ ]:
val_probs = model.predict(val_ds, verbose=0)
val_true = val_df["label"].values

def coverage_curve(probs, y):
    conf = probs.max(1); ok = probs.argmax(1) == y
    ths = np.linspace(0, 0.99, 100)
    cov = np.array([(conf >= t).mean() for t in ths])
    acc = np.array([ok[conf >= t].mean() if (conf >= t).any() else np.nan for t in ths])
    return ths, cov, acc

ths, cov_v, acc_v = coverage_curve(val_probs, val_true)
ok_t = ths[np.nan_to_num(acc_v) >= TARGET_ACCURACY_FOR_THRESHOLD]
THRESHOLD = float(ok_t[0]) if len(ok_t) else 0.9
_, cov_t, acc_t = coverage_curve(probs, y_true)
i = int(np.argmin(np.abs(ths - THRESHOLD)))
print(f"Chosen threshold: {THRESHOLD:.2f}")
print(f"Test set: answers {cov_t[i]*100:.1f}% of photos with {acc_t[i]*100:.1f}% accuracy; "
      f"the other {100-cov_t[i]*100:.1f}% get 'not sure - ask an expert'.")

fig, ax = plt.subplots(figsize=(5.5, 3.6))
ax.plot(ths, acc_t, label="accuracy of answered photos"); ax.plot(ths, cov_t, label="share of photos answered")
ax.axvline(THRESHOLD, color="grey", ls="--"); ax.set_xlabel("confidence threshold"); ax.legend(); ax.set_ylim(0, 1.02)
plt.tight_layout(); plt.savefig(os.path.join(REPORTS_DIR, "threshold.png")); plt.show()

### 10.1 Mistakes to learn from
Looking at wrong predictions shows what kind of photos to collect next.

In [ ]:
wrong = test_eval[~test_eval["correct"]].copy()
wrong["pred_name"] = wrong["pred"].map(lambda k: class_names[k])
wrong["conf"] = probs[~test_eval["correct"].values].max(1)
show_grid(wrong.sort_values("conf", ascending=False).head(18),
          title="Most confident mistakes",
          label_fn=lambda r: f"true: {r['class_name'][:18]}\npred: {r['pred_name'][:18]} ({r['conf']:.0%})")

## 11. Diagnosis demo: photo → result → treatments from the CSV
This is the same logic your app will use: run the model, take the top answers, and look up the
record in `farm_detection_dataset.csv` for signs, prevention and treatment.

In [ ]:
rec_by_id = records.set_index("id")

def diagnose(image_path, top_k=3, threshold=THRESHOLD):
    x, _ = load_image(image_path, 0)
    p = model.predict(x[None], verbose=0)[0]
    order = np.argsort(-p)[:top_k]
    best = order[0]
    result = {"top_predictions": [{"record_id": class_ids[k], "name": class_names[k], "confidence": round(float(p[k]), 4)} for k in order]}
    if p[best] < threshold:
        result["status"] = "not_sure"
        result["message"] = ("The photo is not clear enough for a confident answer. Take another close, sharp photo "
                             "in daylight, or contact your extension officer / vet.")
        return result
    r = rec_by_id.loc[class_ids[best]]
    if r["category"] == "crop_requirement":
        result.update(status="healthy", message=f"This {r['name']} plant looks healthy.",
                      watch_out_for=r["related_ids"])
        return result
    result.update(status="problem_found", record_id=class_ids[best], name=r["name"],
                  scientific_name=r["scientific_name"], severity=r["severity"],
                  key_signs=r["key_signs_for_detection"], prevention=r["prevention"],
                  organic_cultural_control=r["organic_cultural_control"],
                  chemical_control=r["chemical_control"], expert_vet_advice=r["expert_vet_advice"],
                  notifiable=r["notifiable_in_kenya"])
    return result

for path in test_df.groupby("record_id").head(1).sample(3, random_state=SEED)["path"]:
    with Image.open(path) as im:
        plt.figure(figsize=(2.6, 2.6)); plt.imshow(im); plt.axis("off"); plt.show()
    print(json.dumps(diagnose(path), indent=2, ensure_ascii=False)[:1500], "\n")

## 12. Export for the app and save everything
* `models/<name>_best.keras` — full Keras model (for more training later)
* `models/<name>.tflite` — small model for Android / React Native (float16 weights)
* `models/<name>_labels.json` — class index → record ID → name (the app joins this with the CSV)
* `reports/` — metrics, charts and a model card

The TFLite model takes a `224×224×3` float image with **raw pixel values 0–255** and returns
probabilities for each class in the order of the labels file.

In [ ]:
model.save(ckpt2)
export_dir = os.path.join(MODELS_DIR, f"{MODEL_NAME}_savedmodel")
model.export(export_dir)

conv = tf.lite.TFLiteConverter.from_saved_model(export_dir)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.target_spec.supported_types = [tf.float16]
tflite_bytes = conv.convert()
tflite_path = os.path.join(MODELS_DIR, f"{MODEL_NAME}.tflite")
with open(tflite_path, "wb") as f:
    f.write(tflite_bytes)
print(f"TFLite model: {len(tflite_bytes)/1e6:.1f} MB")

# Check the phone model agrees with the full model
interp = tf.lite.Interpreter(model_content=tflite_bytes)
interp.allocate_tensors()
inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
check_paths = test_df.sample(min(200, len(test_df)), random_state=SEED)["path"].values
batch = np.stack([load_image(p, 0)[0].numpy() for p in check_paths]).astype(np.float32)
keras_pred = model.predict(batch, verbose=0).argmax(1)
tflite_pred = []
for x in batch:
    interp.set_tensor(inp["index"], x[None])
    interp.invoke()
    tflite_pred.append(interp.get_tensor(out["index"])[0].argmax())
print(f"TFLite vs Keras agreement on {len(batch)} test photos: {np.mean(np.array(tflite_pred) == keras_pred):.1%}")

In [ ]:
metrics_out = {
    "model_name": MODEL_NAME, "backbone": BACKBONE, "image_size": IMG_SIZE, "num_classes": NUM_CLASSES,
    "train_images": int(len(train_df)), "val_images": int(len(val_df)), "test_images": int(len(test_df)),
    "test_metrics": {k: float(v) for k, v in overall.items()},
    "bootstrap_ci_95": ci_table.set_index("metric")[["ci_95_low", "ci_95_high"]].to_dict("index"),
    "by_source": by_source.set_index("source").to_dict("index"),
    "mcnemar_stage1_vs_stage2": {"only_stage2_right": n10, "only_stage1_right": n01, "p_value": float(p_mc)},
    "expected_calibration_error": float(ece),
    "confidence_threshold": THRESHOLD,
    "skipped_classes_too_few_images": skipped,
}
with open(os.path.join(REPORTS_DIR, f"{MODEL_NAME}_metrics.json"), "w") as f:
    json.dump(metrics_out, f, indent=2, default=float)

card = f"""# Model card: {MODEL_NAME}

* Backbone: {BACKBONE} (ImageNet pre-trained), input {IMG_SIZE}x{IMG_SIZE}, raw pixels 0-255
* Classes: {NUM_CLASSES} ({', '.join(class_ids)})
* Data: {len(train_df):,} train / {len(val_df):,} validation / {len(test_df):,} test images
* Sources: {', '.join(sorted(images['source'].unique()))}
* Test accuracy: {overall['accuracy']:.3f} (95% CI {ci_table.iloc[0]['ci_95_low']:.3f}-{ci_table.iloc[0]['ci_95_high']:.3f}), macro F1 {overall['macro_f1']:.3f}
* Field-photo results: see by_source in the metrics file - these matter most for farmers
* "Not sure" threshold: {THRESHOLD:.2f}

## Limitations
* Many training photos are lab photos (PlantVillage); accuracy on real farm photos is lower.
* Only classes with at least {MIN_TRAIN_IMAGES} training photos are included; skipped: {', '.join(skipped) or 'none'}.
* Treatment advice comes from farm_detection_dataset.csv and must be reviewed by an agronomist / vet.
* Not tested on Kilifi field photos yet - collect and add them, then retrain.
"""
with open(os.path.join(REPORTS_DIR, "MODEL_CARD.md"), "w") as f:
    f.write(card)
print(card)

In [ ]:
# Copy results (models, reports, updated CSV) back to Google Drive so they survive Colab disconnects
if OUTPUT_ON_DRIVE:
    stamp = time.strftime("%Y%m%d_%H%M")
    dest = os.path.join(OUTPUT_ON_DRIVE, f"{MODEL_NAME}_{stamp}")
    shutil.copytree(MODELS_DIR, os.path.join(dest, "models"))
    shutil.copytree(REPORTS_DIR, os.path.join(dest, "reports"))
    shutil.copy(CSV_PATH, dest)
    print("Saved to Google Drive:", dest)

## Next steps
1. **Add your own Kilifi photos** to `data/images/<ID>_.../` (confirmed by an extension officer or vet),
   run `python scripts/update_images.py`, re-zip and run this notebook again.
2. **Animals and soils:** once you have at least 30+ photos per class, set
   `CATEGORIES_TO_TRAIN = ["animal_disease", "animal_pest"]` (or `["soil_type"]`) and a new
   `MODEL_NAME`, and train a separate model.
3. Watch the **field-photo** accuracy and the per-class recall table — those tell you which
   classes need more real photos.
4. In the app: load the `.tflite` model + labels file, and show the CSV advice for the predicted
   record ID. Below the confidence threshold, show "not sure — contact an expert".